In [1]:
#easy -riya
!pip install qiskit qiskit-aer

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Create a 2-qubit circuit
qc = QuantumCircuit(2, 2)

# Step 1: Prepare a Bell pair
qc.h(0)
qc.cx(0, 1)

# Step 2: Encode message '00'
# No gates are required for message 00

# Step 3: Decode the message
qc.cx(0, 1)
qc.h(0)

# Step 4: Measure both qubits
qc.measure([0, 1], [0, 1])

# Simulate the circuit
simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print("Measurement results:", counts)

if '00' in counts:
    print("Message '00' decoded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.9 MB/s eta 0:00:00
Measurement results: {'00': 1024}
Message '00' decoded successfully!


In [2]:
#medium
!pip install qiskit qiskit-aer

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()
messages = ["00", "01", "10", "11"]

print("Superdense Coding Results\n")

for message in messages:
    qc = QuantumCircuit(2, 2)

    # Step 1: Create an entangled Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Step 2: Encode the 2-bit message
    # First bit controls Z, second bit controls X
    if message[0] == "1":
        qc.z(0)
    if message[1] == "1":
        qc.x(0)

    # Step 3: Decode at receiver
    qc.cx(0, 1)
    qc.h(0)

    # Step 4: Measure both qubits
    qc.measure([0, 1], [0, 1])

    # Step 5: Simulate
    result = simulator.run(qc, shots=1024).result()
    counts = result.get_counts()

    decoded = max(counts, key=counts.get)

    print(f"Sent: {message} | Decoded: {decoded} | Counts: {counts}")

Superdense Coding Results

Sent: 00 | Decoded: 00 | Counts: {'00': 1024}
Sent: 01 | Decoded: 10 | Counts: {'10': 1024}
Sent: 10 | Decoded: 01 | Counts: {'01': 1024}
Sent: 11 | Decoded: 11 | Counts: {'11': 1024}


In [3]:
#hard
!pip install qiskit qiskit-aer

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

message = "00"
shots = 1024

# Test without and with a bit-flip error
for error in [False, True]:

    qc = QuantumCircuit(2, 2)

    # Step 1: Create Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Step 2: Encode message
    if message[0] == "1":
        qc.z(0)
    if message[1] == "1":
        qc.x(0)

    # Step 3: Simulate transmission bit-flip error
    if error:
        qc.x(0)

    # Step 4: Decode message
    qc.cx(0, 1)
    qc.h(0)

    # Step 5: Measure
    qc.measure([0, 1], [0, 1])

    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()

    correct = counts.get(message, 0)
    accuracy = correct / shots * 100

    print("\nBit-flip error:", error)
    print("Measurement results:", counts)
    print(f"Decoding accuracy: {accuracy:.2f}%")


Bit-flip error: False
Measurement results: {'00': 1024}
Decoding accuracy: 100.00%

Bit-flip error: True
Measurement results: {'10': 1024}
Decoding accuracy: 0.00%


In [ ]:
#challenge
!pip install qiskit qiskit-aer matplotlib

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# Set target Bell-pair fidelity
fidelity = 0.99
shots = 2000

# For Qiskit's 1-qubit depolarizing channel:
# Bell-state fidelity F = 1 - 3p/4
p = 4 * (1 - fidelity) / 3

# Create noise model
noise_model = NoiseModel()
error = depolarizing_error(p, 1)
noise_model.add_all_qubit_quantum_error(error, ['id'])

simulator = AerSimulator(noise_model=noise_model)

messages = ['00', '01', '10', '11']
total_correct = 0
total_shots = 0

print("Target Bell-pair fidelity:", fidelity)
print("\nSuperdense Coding with Noise\n")

for message in messages:
    qc = QuantumCircuit(2, 2)

    # Step 1: Create Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Step 2: Model noise on the shared entangled pair
    qc.id(0)

    # Step 3: Encode the two classical bits
    if message[0] == '1':
        qc.x(0)
    if message[1] == '1':
        qc.z(0)

    # Step 4: Decode at receiver
    qc.cx(0, 1)
    qc.h(0)

    # Step 5: Measure
    qc.measure([0, 1], [0, 1])

    # Preserve the identity gate so its noise is simulated
    compiled = transpile(
        qc, simulator, optimization_level=0
    )

    result = simulator.run(
        compiled, shots=shots
    ).result()

    counts = result.get_counts()
    correct = counts.get(message, 0)
    accuracy = correct / shots * 100

    total_correct += correct
    total_shots += shots

    print(f"Sent message: {message}")
    print("Measurement results:", counts)
    print(f"Accuracy: {accuracy:.2f}%\n")

overall_accuracy = total_correct / total_shots * 100
error_rate = 100 - overall_accuracy

print(f"Overall decoding accuracy: {overall_accuracy:.2f}%")
print(f"Overall decoding error rate: {error_rate:.2f}%")
